# Assignment 3: Assemble IR and LM for RAG

## Overview
Assemble the BM25 retriever from Assignment 1 and the text generator from Assignment 2 to build a RAG system. Test it with open-form questions on the Cranfield collection. This assignment covers four tasks: building the RAG system, selecting the best generator, understanding how retrieval affects output quality, and diagnosing where the pipeline succeeds or fails. All evaluations in this assignment are qualitative. In your analysis, do not just describe what the model outputs, but engage with why it behaves the way it does and what it reveals about the system.

## Requirements
- Make sure all cells have been run and outputs are visible. Queries and model responses **must be displayed** in the notebook output.
- Implement each task in the cells marked **TODO** and answer questions marked **TODO**

## Deadline
**May 1, 23:59 CET**

## Submission
- Submit only the .ipynb file. Add your name to the filename.

In [1]:
# Load imports
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import torch
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForCausalLM
import ir_datasets


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the models and tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Replace these paths with your actual Assignment 2 checkpoints
#model_full   = AutoModelForCausalLM.from_pretrained("ckpt-full") #Given
model_full   = AutoModelForCausalLM.from_pretrained("/Users/merterol/Desktop/uzh_code/uzh/Computational Linguistics/Master/RAG/Assignment 2/ckpt-full")

#model_completion = AutoModelForCausalLM.from_pretrained("ckpt-completion") #Given
model_completion = AutoModelForCausalLM.from_pretrained("/Users/merterol/Desktop/uzh_code/uzh/Computational Linguistics/Master/RAG/Assignment 2/ckpt-completion")

print("Both models loaded.")

Both models loaded.


## Task 1: Build a RAG System

### Task overview
1. Implement a `BM25` class (ported from Assignment 1) and build an index over the Cranfield collection.
2. Implement a `RAGSystem` class that ties retrieval and generation together.




### Task 1.1: BM25 Retriever

Port your `BM25` implementation from Assignment 1, keeping only the attributes and methods you require.

In [3]:
@dataclass
class BM25:
    """

    Brute-force scoring over all documents (no inverted index).
    """
    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        
        # --- BM25 Hyperparameters ---
        self.k1: float = k1                    # term frequency saturation
        self.b: float = b                      # document length normalization
        self.remove_stopwords: bool = remove_stopwords
        
        # Initializations for tokenization
        self._stemmer = PorterStemmer()      # work stemming
        self._stopwords = set(stopwords.words("english")) if self.remove_stopwords else set() # stopwords set
        
        # --- Corpus Statistics (populated by build()) ---
        self.N: int = 0                        # total number of documents
        self.avgdl: float = 0.0               # average document length
        self.df: Dict[str, int] = {}          # term -> number of docs containing term
        self.doc_len: Dict[str, int] = {}     # doc_id -> document length
        self.doc_tf: Dict[str, Counter] = {}  # doc_id -> term frequency Counter
        self.docs_store: Dict[str, Dict[str, str]] = {}  # doc_id -> {"title": ..., "text": ...}

    def tokenize(self, text: str) -> List[str]:
        """
        Convert raw text into a list of processed tokens.

        Requirements:
          1. Lowercase everything         e.g.  "UZH"                       -> ["uzh"]
          2. Extract alphanumeric words   e.g.  "hello, world 2026!"        -> ["hello", "world", "2026"]
          3. Remove stopwords             e.g.  "the document has a title"  -> ["document", "title"]
          4. Apply Porter stemming        e.g.  "working worked works"      -> ["work", "work", "work"]
          5. Drop empty tokens            e.g.  " retrieval "               -> ["retrieval"]

        Args:
            text: raw input string
        Returns:
            List of processed token strings

        Hints:
            - Perform text normalization using Python string methods.
            - Use the `re` module to extract alphanumeric tokens.
            - For stemming and stopword removal, use the initialized stemmer and stopword list from the BM25 class constructor.

        """
        tokenizer = nltk.RegexpTokenizer(r'\w+')
        tokens = tokenizer.tokenize(text.lower())
        stop_words = set(stopwords.words('english'))

        filtered_tokens = [self._stemmer.stem(token) for token in tokens if token not in stop_words]
        return filtered_tokens
    
    def build(self, docs_iter: Iterable) -> None:
        counter = 0
        len_docs = 0

        # Step 1-5:
        for doc in docs_iter:        
            counter += 1

            title = doc.title or ""
            text = doc.text or ""
            title_and_text = title + " " + text

            tokens = self.tokenize(title_and_text)
            len_docs += len(tokens)

            self.docs_store[doc.doc_id] = {"title": title, "text": text}
            self.doc_len[doc.doc_id] = len(tokens)
            self.doc_tf[doc.doc_id] = Counter(tokens)

        # step 6:
        for doc_id, tf_counter in self.doc_tf.items():
            for term in tf_counter:
                self.df[term] = self.df.get(term, 0) + 1

        # step 7:
        self.N = counter
        self.avgdl = len_docs / counter
        
    def idf(self, term: str) -> float:
        df = self.df.get(term, 0)
    
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)
    
    def score(self, query_tokens: List[str], doc_id: str) -> float:
        if doc_id not in self.doc_len or self.doc_len[doc_id] == 0:
            return 0.0
    
        score = 0.0
        dl = self.doc_len[doc_id]
        tf = self.doc_tf[doc_id]

        for term in set(query_tokens):
            f = tf.get(term, 0)

            if f == 0:
                continue
            
            df = self.df.get(term, 0)
            idf = math.log(1 + (self.N - df + 0.5) / (df + 0.5))

            denom = f + self.k1 * (1 - self.b + self.b * (dl / self.avgdl))

            score += idf * (f * (self.k1 + 1)) / denom

        return score
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
        tokenized = self.tokenize(query_text)
        docs = []

        for doc_id in self.doc_tf.keys():
            score = self.score(tokenized, doc_id)
            if score > 0:
                docs.append((doc_id, score))

        # Sort documents by score in descending order
        docs.sort(key=lambda x: x[1], reverse=True)

        return docs[:k]
    def evaluate(self, queries_iter: Iterable, qrels_iter: Iterable, k: int = 10, metrics: Optional[List[str]] = None) -> Dict[str, float]:
        if metrics is None:
            metrics = ["recall", "precision", "mrr", "ndcg"]

        def normalize_relevance(rel):
            return 1 if rel > 0 else 0

        def dcg(rels):
            return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))

        qrels_dict = defaultdict(dict) # qrels[query_id][doc_id] = normalized relevance

        for qrel in qrels_iter:
            qid = qrel.query_id
            doc_id = qrel.doc_id
            rel = normalize_relevance(qrel.relevance)

            if doc_id in qrels_dict[qid]:
                qrels_dict[qid][doc_id] = max(qrels_dict[qid][doc_id], rel)
            else:
                qrels_dict[qid][doc_id] = rel

        results = {metric: [] for metric in metrics}

        for query in queries_iter:
            qid = query.query_id
            qtext = query.text

            if qid not in qrels_dict:
                continue
            
            relevant_docs = {doc_id: rel for doc_id, rel in qrels_dict[qid].items() if rel > 0}
            num_relevant = len(relevant_docs)

            retrieved_docs = self.retrieve(qtext, k)

            retrieved_doc_ids = [doc_id for doc_id, _ in retrieved_docs]
            retrieved_rels = [relevant_docs.get(doc_id, 0) for doc_id in retrieved_doc_ids]

            if "recall" in metrics:
                recall = sum(retrieved_rels) / num_relevant if num_relevant > 0 else 0.0
                results["recall"].append(recall)

            if "precision" in metrics:
                precision = sum(retrieved_rels) / k
                results["precision"].append(precision)

            if "mrr" in metrics:
                mrr = 0.0
                for rank, rel in enumerate(retrieved_rels, 1):
                    if rel > 0:
                        mrr = 1 / rank
                        break
                results["mrr"].append(mrr)

            if "ndcg" in metrics:
                ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
                idcg = dcg(ideal_rels)
                ndcg = dcg(retrieved_rels) / idcg if idcg > 0 else 0.0
                results["ndcg"].append(ndcg)

        avg_results = {}
        for metric in metrics:
            values = results[metric]
            avg_results[metric] = sum(values) / len(values) if values else 0.0

        return avg_results
    def grid_search(self, queries_iter_factory, qrels_iter_factory, k: int = 10, k1_values: Tuple = (0.60, 1.20, 1.80), b_values: Tuple = (0.25, 0.50, 0.75)) -> List[Dict]:
        results = []

        for k1 in k1_values:
            for b in b_values:
                self.k1 = k1
                self.b = b

                eval_results = self.evaluate(queries_iter_factory(), qrels_iter_factory(), k)
                eval_results.update({"k1": k1, "b": b})
                results.append(eval_results)

        return results



### Build the Cranfield Index

Load the Cranfield corpus with `ir_datasets` and call `bm25.build()`. You may use the best `k1` and `b` hyperparameters from Assignment 1.


In [4]:
# Load the Cranfield corpus and build the BM25 index
dataset = ir_datasets.load("cranfield")
#TODO
bm25 = BM25(k1=1.8, b=0.75)   # <- use your best k1/b from Assignment 1
bm25.build(dataset.docs_iter())

print(f"Index built: {bm25.N} documents, avg doc length = {bm25.avgdl:.1f} tokens")


Index built: 1400 documents, avg doc length = 102.9 tokens


### Task 1.2: RAGSystem

Implement `RAGSystem` with the following behaviour:

* **`build_prompt(query, retrieved_docs)`** — assemble the prompt using the **same instruction-tuning template** as in Assignment 2 (`format_prompt`). Concatenate the retrieved documents as a single doc to accomodate *k* larger than 1.
* **`generate(query, k, verbose)`** — full RAG pipeline:
  1. Retrieve top-*k* docs via `self.bm25.retrieve()`
  2. Build the prompt with `build_prompt`
  3. Run inference; **strip the prompt prefix** from the decoded output
  4. Return the answer string only
  5. Have a *verbose* flag to return the retrieved docs too. It will be useful for evaluation.
The constructor must accept `model`, `tokenizer`, `bm25`, `k` (default 3), and `max_new_tokens` (default 256).


In [5]:
class RAGSystem:
    """
    Retrieval-Augmented Generation (RAG) pipeline combining BM25 retrieval
    with a fine-tuned causal language model for open-form question answering.

    The pipeline operates in two stages:
      1. Retrieval: the BM25 index is queried to fetch the top-k most
         relevant documents for a given question.
      2. Generation: the retrieved documents are injected into an
         instruction-tuning prompt and passed to the language model, which
         produces a grounded answer.

    Typical usage:
        rag = RAGSystem(model=model, tokenizer=tokenizer, bm25=bm25)
        answer = rag.generate("What causes boundary layer separation?", k=3)
    """

    def __init__(
        self,
        model,
        tokenizer,
        bm25: BM25,
        max_new_tokens: int = 500,
    ):
        """
        Initialise the RAG system.

        Args:
            model          : fine-tuned causal language model used for generation.
            tokenizer      : tokenizer corresponding to *model*
            bm25           : a fully built BM25 instance (``bm25.build()`` must
                             have been called before passing it here).
            max_new_tokens : maximum number of new tokens the model may generate
                             per call to :meth:`generate`.
        """
        self.model = model
        self.tokenizer = tokenizer
        self.bm25 = bm25
        self.max_new_tokens = max_new_tokens
        # self.device = "cuda" if torch.cuda.is_available() else "cpu" # Given
        self.device = "mps" if torch.backends.mps.is_available() else self.device # Mine
        self.model.to(self.device)

    def build_prompt(
        self,
        query: str,
        retrieved_docs: list,
        instruction: str = None,
    ) -> str:
        """
        Assemble the instruction-tuning prompt used for RAG generation.

        Follows the same three-block template as Assignment 2's ``format_prompt``:
        ``### Instruction``, ``### Input``, ``### Response``.  
        
        Retrieved documents are concatenated (title + body) and inserted as the
        ``Document`` field of the input block, allowing the model to draw on
        multiple passages in a single forward pass.

        **Tip**: If you are unsure what the prompt should look like, inspect a
        few samples from your Assignment 2 test set — the structure used there
        is exactly what this method should replicate (minus the retrieved
        context, which replaces the original document field).

        Args:
            query         : the user question or instruction to be answered.
            retrieved_docs: list of document dicts, each containing ``"title"``
                            and ``"text"`` keys (as stored in ``BM25.docs_store``).
            instruction   : optional override for the ``### Instruction`` block.
                            Defaults to a generic technical-QA instruction when
                            *None*. Look at Test data samples for sample instructions

        Returns:
            Formatted prompt string ready for tokenisation, ending with
            ``"### Response:\\n"`` so the model continues from that position.
        """
        if instruction is None:
            instruction = "Answer the following technical question using only the information in the document."
        
        # Concat retrieved documents into a single string
        for doc in retrieved_docs:
            title = doc.get("title", "")
            text = doc.get("text", "")
            query += f"\n\nDocument:\nTitle: {title}\n{text}"
        
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{query}\n\n"
            f"### Response:\n"
        )
        
        return prompt

    def generate(
        self,
        query: str,
        k: int = 3,
        verbose: bool = False,
    ) -> str | tuple:
        """
        Run the full RAG pipeline for a single query.

        Steps:
          1. Retrieve the top-k documents from the BM25 index.
          2. Build the instruction-tuning prompt via `build_prompt`.
          3. Tokenise the prompt and run generation with the language model.
          4. Decode only the newly generated tokens (the prompt prefix is stripped).
          5. Return the answer, optionally alongside the retrieved documents.

        Args:
            query   : raw (un-tokenised) user question.
            k       : number of documents to retrieve and include in the prompt.
            verbose : when *True*, returns a ``(answer, retrieved_docs)`` tuple
                      so callers can inspect which passages informed the answer.

        Returns:
            The generated answer string when verbose is False, or a
            ``(answer, retrieved_docs)`` tuple when verbose is True.
        """
        # Step 1:
        retrieved_docs = []
        retrieved_doc_tuples = self.bm25.retrieve(query, k)
        for doc_id, score in retrieved_doc_tuples:
            retrieved_docs.append(self.bm25.docs_store[doc_id])
            
        # Step 2:
        prompt = self.build_prompt(query, retrieved_docs)
        
        # Step 3:
        input_tokens = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        
        # Step 4:
        with torch.no_grad():
            output_tokens = self.model.generate(
                **input_tokens,
                max_new_tokens=self.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
            )
            
        generated_tokens = output_tokens[0][input_tokens["input_ids"].shape[1]:]
        answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        return (answer, retrieved_docs) if verbose else answer


### Task 1.3: Check your implementation

Run the cell below to verify that the full RAG pipeline works end-to-end.  
**Note:** this cell requires a loaded model and tokenizer — run it after you have loaded at least one checkpoint in Problem 2.

In [6]:
# Full RAG test
# Replace model / tokenizer with whichever checkpoint you loaded first.
SAMPLE_QUERY = "what are the structural and aeroelastic problems associated with flight of high speed aircraft"

rag_test = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

answer, docs = rag_test.generate(SAMPLE_QUERY, k=1, verbose=True)

print("=== Retrieved documents ===")
for i, doc in enumerate(docs, 1):
    title = doc.get("title", "").strip() or "(no title)"
    snippet = doc.get("text", "").replace("\n", " ")
    print(f"[{i}] {title}\n    {snippet}...\n")

print("=== Generated answer ===")
print(answer)

=== Retrieved documents ===
[1] some structural and aerelastic considerations of high
speed flight .
    some structural and aerelastic considerations of high speed flight .   the dominating factors in structural design of high-speed aircraft are thermal and aeroelastic in origin .  the subject matter is concerned largely with a discussion of these factors and their interrelation with one another .  a summary is presented of some of the analytical and experimental tools available to aeronautical engineers to meet the demands of high-speed flight upon aircraft structures .  the state of the art with respect to heat transfer from the boundary layer into the structure, modes of failure under combined load as well as thermal inputs and acrothermoelasticity is discussed .  methods of attacking and alleviating structural and aeroelastic problems of high-speed flight are summarized .  finally, some avenues of fundamental research are suggested ....

=== Generated answer ===
The document descr

---
# Tasks 2–4: Evaluation & Analysis

The remainder of this assignment is about evaluating and analysing the outputs of your RAG system by systematically varying inputs — such as queries, prompt instructions, and retrieval depth — and observing how the outputs change.

## Query Selection

<p style="font-size:1.2em">Manually select queries from the <strong>held-out test set</strong> of your Assignment 2 instruction-tuning data. Do <strong>not</strong> randomly sample from the full Cranfield dataset. If you modify or expand a query (e.g. for better retrieval), you may do so but must describe exactly how you did it. A single query may be reused across <strong>at most two</strong> tasks.</p>

Queries must come from the test set to avoid **data leakage** — if a query was seen during fine-tuning, the model may reproduce a memorised answer rather than reasoning from the retrieved context, making your evaluation unreliable.

---

## Task 2: Generator Evaluation

Compare your two fine-tuned models from Assignment 2 to choose the best generator for your system:
- **RAG A** — fine-tuned with loss over the **full prompt** (instruction + input + response)
- **RAG B** — fine-tuned with loss over the **completion only** (response tokens only)

**Note**: This task evaluates the generator in isolation. Both models receive the same retrieved context. Evaluate the generated responses only, not the quality of the retrieval.

### Implement RAG systems with both the models

In [7]:
# Instantiate one RAGSystem per model using the same BM25 index
rag_A = RAGSystem(model=model_full, tokenizer=tokenizer, bm25=bm25)
rag_B = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

### Part 1: Select Queries

Choose queries covering **all three** conditions below. Fill in your final chosen queries in the code cell (TODO).

| Condition | Minimum | Your queries |
|-----------|---------|---------------|
| Simple factual query | 2 | |
| Query requiring a structured/formatted answer (e.g. bullet points) | 2 | |
| One query tested with two different reformulations (same keywords, different wording / instruction template) | 1 pair | |



In [ ]:
# TODO: Run both models and print out Queries and Responses
# You are free to choose parameters such as k, but the same parameters must be used for both models.

### Part 2: Evaluate the outputs of both models using the following criteria where applicable

Use the criteria below to evaluate your outputs and write down your observations.

| Criterion | Description |
|-----------|-------------|
| **Completeness** | Does the answer address the entire query given the available context? |
| **Instruction Following** | Does the model follow the required format / instructions? |
| **Robustness** | Is the model consistent across different reformulations of the same query? |
| **Fluency** | Is the output coherent and readable? |


#### Evaluation Table

TODO: Fill out the observation column on how the models behave. 

| Query type | Model | Observation |
|------------|-------|-------------|
| Factual | Completion-only | |
| | Full prompt | |
| Structured | Completion-only | |
| | Full prompt | |
| Reform (v1 vs v2) | Completion-only | |
| | Full prompt | |

### Part 3: TODO: Based on your evaluations, answer the following questions

**Q1. Is there a query type where one model clearly outperforms the other?**

...

**Q2. Which model do you select as the better generator, and what is your justification?**

...

**Q3. Does your selection align with the model you expected to perform better based on PPL_resp and PPL_all from Assignment 2? Discuss any discrepancy.**

...


## Task 3: RAG System Evaluation

This task compares your selected model in two settings: with RAG and without retrieval (no-RAG). No-RAG serves as the baseline, and the model receives only the query with instruction and no retrieved context.

- **RAG**: top-*k* Cranfield documents are retrieved and injected into the prompt.
- **No-RAG**: the model answers from parametric knowledge only (no retrieved context).

The goal is to assess how much access to external context improves factual accuracy and reduces hallucination. Use the model you selected at the end of Task 2.

> **Note**: For RAG outputs, inspect the retrieved documents alongside the generated answer. You will need them to evaluate groundedness and use of context.

> **Important**: The no-RAG baseline must use the same `### Instruction` / `### Input` / `### Response` prompt template as the RAG setting, with the retrieved documents omitted. Passing the query as a bare string would turn it into a plain autocompletion prompt, which is not a fair comparison.

In [ ]:
# Sample function for generating noRAG responses. You are allowed to edit as you wish 
def generate_no_rag(query: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Generate an answer using the language model without any retrieved context.

    Serves as the no-RAG baseline: the model receives only the query wrapped
    in the standard instruction-tuning template and must rely entirely on its
    parametric (fine-tuned) knowledge to produce an answer.

    Args:
        query          : raw user question to answer.
        model          : causal language model to use for generation.
        tokenizer      : tokenizer corresponding to *model*.
        max_new_tokens : maximum number of new tokens to generate.

    Returns:
        Decoded answer string with the prompt prefix stripped and leading/
        trailing whitespace removed.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    prompt = (
        "### Instruction:\nAnswer the following technical question.\n\n"
        f"### Input:\nQuestion: {query}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()



### Part 1: Select Queries

Select **at least 5** queries with at least one per condition. Avoid reusing the same set of queries used in Task 2.

| Condition | Description | Your query (TODO)|
|-----------|-------------|------------|
| Simple | Does not require specific domain knowledge | |
| Knowledge-intensive| Requires specific or detailed factual knowledge | |
| Complex / multi-part | Concatenate two questions | |
| Comparative | Ask for similarities or differences between concepts | |
| Out-of-domain | A random query outside the Cranfield aeronautics domain (e.g. What are the components of ibuprofen?) | |


In [ ]:
#TODO: Print out the Queries and Response

# Assign your chosen model from Problem 2
rag_system = RAGSystem(model=model_chosen, tokenizer=tokenizer, bm25=bm25)

### Part 2: Evaluate Outputs


| Criterion | Description |
|-----------|-------------|
| **Groundedness** | Is the answer supported and constrained by the retrieved context? |
| **Factual Accuracy** | Is the information correct? |
| **Hallucination** | Does the model introduce unsupported or incorrect information? |
| **Refusal** | Does the model acknowledge uncertainty or decline to answer? |
| **Use of Context** | Does the model effectively incorporate relevant retrieved details? |



#### Evaluation Table

TODO: For each query type and setting, write 1–2 concise sentences describing your observations of the model's output. Base your observations on the evaluation criteria listed above. You are free to add extra evaluation criteria.

| Query type | Setting | Observation (TODO) |
|------------|---------|-------------|
| Simple | RAG | |
| | No-RAG | |
| Knowledge-intensive | RAG | |
| | No-RAG | |
| Complex | RAG | |
| | No-RAG | |
| Comparative | RAG | |
| | No-RAG | |
| Out-of-domain | RAG | |
| | No-RAG | |

### Part 3: TODO: Based on your evaluations, answer the following questions

**Q1. Overall, does RAG improve over no-RAG? Summarize your findings.**

...

**Q2. For which query types does RAG provide the most noticeable improvement, and why?**

...

**Q3. How much does retrieval affect hallucination? Does it consistently reduce hallucinations, or are there cases where it has little or no effect (or even introduces errors)? Discuss with examples.**

...

**Q4. How does the overall quality of answers differ between RAG and no-RAG? Consider not just fluency but also substance and filler language.**

...

**Q5. How does the model behave on the out-of-domain query with and without RAG? Which behavior is more desirable from a user's perspective? Explain why.**

...


## Task 4: Error Analysis

Analyse when the RAG pipeline succeeds or fails, and identify where failures originate — retrieval, generation, or both.


### Part 1: Identify One Example per Case

Using open-form queries, find **at least one example** of each case below. For each case, 

(a) determine whether the root cause is retrieval, generation, or both,

(b) propose a solution for the error cases.

| Case | Description | Diagnostic questions |
|------|-------------|----------------------|
| **Refusal** | The model declines to answer or expresses uncertainty | Was the context relevant and sufficient? Did the model fail to use available information? |
| **Incorrect Answer** | The model provides a wrong or unsupported answer | Was the context relevant? Did the model correctly use the retrieved information? |
| **Correct Answer** | The model provides a correct and relevant answer | Was the context relevant? Did the model correctly use the retrieved information? |


In [ ]:
#TODO: Print out the Queries and Response
# Find queries that illustrate each case.
# Experiment until you observe each behaviour.



#### TODO: Case Analysis

**Refusal case:**
- Query:
- Was the retrieved context relevant and sufficient?
- Did the model fail to use available information?
- Root cause (retrieval / generation / both):
- Proposed solution:

---

**Incorrect answer case:**
- Query:
- Was the retrieved context relevant and sufficient?
- Did the model correctly use the retrieved information?
- Root cause (retrieval / generation / both):
- Proposed solution:

---

**Correct answer case:**
- Query:
- Was the retrieved context relevant and sufficient?
- Did the model correctly use the retrieved information?
- Root cause (retrieval / generation / both):


### Part 2: Experiment with at least two additional values of $k$

Select one query where retrieval was the cause of failure and one where the generator was the cause. 

Experiment at least two additional values of $k$ for each (one higher and one lower than your original setting) and discuss the results. 



In [ ]:
#TODO: Experiment and print out the queries and response. Mention the k values


#### TODO: Discussion

**Does increasing or decreasing $k$ resolve the error, introduce new ones, or make no difference? What does this suggest about the advantages and limitations of retrieval depth?**

...
